# Plan 6 Phase B — Single-A100 Run (`PatchTST`, `Chronos`, `FinBERT`, `Qwen/Qwen3.8-27B` 4-bit QLoRA)

Plan 6 lives in `plan_6_stock_volatility/`. This notebook clones or pulls `main`, verifies an **NVIDIA A100 (40GB / 80GB)** runtime, installs the Phase B GPU dependencies (`transformers` supporting `qwen3_5`, `peft`, `bitsandbytes`, `accelerate`, `chronos-forecasting`), runs `scripts/colab_phase_b.py --allow-weight-download` against the **frozen Phase A bundle** (`data/frozen/`, read-only), prints the 2025 H3 comparison table, and packages `outputs/phase_b/` for download back into the repository.

- Default 27B LLM: **`Qwen/Qwen3.8-27B`** (`Qwen3_5ForConditionalGeneration`, ungated, text-only LoRA `r=16` on `model.language_model` excluding `visual`/`mtp`).
- Optional fallback flag if desired: `--llm-model Qwen/Qwen3-32B` or `--llm-model Qwen/Qwen3-30B-A3B`.
- See `PHASE_B.md` for full specifications and post-Colab CS482 final PDF integration (`scripts/07_integrate_phase_b_report.py`).

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Lc-wing/data_science_project.git"
BRANCH = "main"
PLAN_DIR = "plan_6_stock_volatility"
ENTRY = "scripts/colab_phase_b.py"

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)

top = Path("data_science_project").resolve()
if (top / ".git").exists():
    for args in (("fetch", "origin", BRANCH), ("checkout", BRANCH), ("pull", "origin", BRANCH)):
        done = git(*args, cwd=top)
        assert done.returncode == 0, done.stderr
else:
    done = git("clone", "--branch", BRANCH, REPO, str(top))
    assert done.returncode == 0, done.stderr

plan_root = top / PLAN_DIR
assert (plan_root / ENTRY).exists(), f"{ENTRY} not found under {plan_root}"
os.chdir(plan_root)
print("Working directory:", Path.cwd())


In [ ]:
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to an NVIDIA A100 (40GB/80GB) GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# Install Phase B GPU stack (latest transformers for qwen3_5 / Qwen/Qwen3.8-27B support)
!pip install -q -U "transformers>=4.57" peft bitsandbytes accelerate chronos-forecasting
!pip install -q -U pyarrow lightgbm arch shap
!pip install -q -e . --no-deps


In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Run full Phase B: PatchTST 1M/15M + Chronos + FinBERT + Qwen/Qwen3.8-27B Zero-Shot & QLoRA
!python scripts/colab_phase_b.py --allow-weight-download --llm-model Qwen/Qwen3.8-27B --llm-epochs 1


In [ ]:
import shutil
from pathlib import Path

print(Path("outputs/phase_b/phase_b_comparison.md").read_text())
archive = shutil.make_archive(
    "phase_b_a100_outputs", "zip", root_dir=".", base_dir="outputs/phase_b"
)
print("Packaged archive for download ->", archive)
